# UTS PBL Machine Learning — SmartReceipt Analyzer
## Model: Multinomial Naive Bayes

**Anggota:** Rakagali

Notebook ini merupakan bagian dari laporan UTS kelompok. Seluruh anggota memakai
**dataset, pra-pengolahan, pembagian data, dan ekstraksi fitur TF-IDF yang identik**.
Yang berbeda hanya algoritma klasifikasinya, sehingga hasil evaluasi dapat
dibandingkan secara adil.

Alur kerja:

```text
Teks struk OCR -> Pra-pengolahan -> TF-IDF (char n-gram) -> Multinomial Naive Bayes -> Evaluasi
```

### 1. Deskripsi Data
Dataset yang digunakan:
- `Dataset_SmartReceipt_Sintetis.csv` — 6.026 baris teks item struk sintetis dengan 15 kategori.
- `Dataset_Katalog_Indonesia_Sintetis_v1.csv` — 1.190 baris katalog produk retail Indonesia sebagai data latih tambahan.

Kolom utama:
| Kolom | Keterangan |
|---|---|
| `nama_produk` | Nama baku produk |
| `receipt_text` | Teks menyerupai hasil OCR struk (fitur input) |
| `kategori` | Label target (15 kategori) |
| `split` | Pembagian `train` / `val` / `test` |

Pembagian data memakai kolom `split` bawaan dengan *leakage guard*: produk pada
split validasi/uji tidak pernah muncul di data latih.


### Import Library & Muat Data

In [ ]:
import sys
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
for base in list(candidates):
    candidates += [base / "ML" / "uts", base / "uts"]
for base in candidates:
    if (base / "pipeline_utils.py").exists():
        sys.path.insert(0, str(base))
        break

import numpy as np
import pandas as pd
from IPython.display import display

import pipeline_utils as pu

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("Konfigurasi TF-IDF:", pu.TFIDF_CONFIG)


In [ ]:
train, val, test = pu.load_splits()

print(f"Data latih : {train.shape[0]:,} baris")
print(f"Data val   : {val.shape[0]:,} baris")
print(f"Data uji   : {test.shape[0]:,} baris")
print(f"Jumlah kategori target: {train['kategori'].nunique()}")


In [ ]:
ringkasan = pd.DataFrame({
    "Jumlah Baris": [len(train), len(val), len(test)],
    "Jumlah Produk Unik": [
        train["nama_produk"].nunique(),
        val["nama_produk"].nunique(),
        test["nama_produk"].nunique(),
    ],
}, index=["Latih", "Validasi", "Uji"])
display(ringkasan)

display(train["kategori"].value_counts().rename("Jumlah Sampel Latih").to_frame())

contoh = train[["nama_produk", "receipt_text", "text", "kategori"]].sample(8, random_state=pu.SEED)
display(contoh)


### 2. Pra Pengolahan Data

Tahapan yang diterapkan (identik untuk semua anggota):

1. **Case folding** — seluruh teks menjadi huruf kecil.
2. **Pembersihan nominal & satuan** — menghapus harga (`Rp 4.500`), ukuran kemasan (`85G`, `600ML`), dan kode PLU/barcode.
3. **Pembersihan simbol** — menghapus karakter non-alfanumerik dan merapikan spasi.
4. **Normalisasi merek/sinonim ke kata generik** — mis. `indomie` -> `mie instan`, `aqua` -> `air mineral`, `pertamax` -> `bensin`.
5. **Pembuangan stopword & token terlalu pendek**.


In [ ]:
prep = pu.TextPreprocessor()

contoh_teks = [
    "INDOMIE GRG SPL 85G",
    "AQUA BTL 600ML Rp 4.500",
    "635110203830 INDOMIE AYAM BAWAN",
    "PARACETAMOL 2PCS",
    "PERTAMAX 5L",
]

for teks in contoh_teks:
    print(f"{teks!r:45} -> {prep.normalize(teks)!r}")


### 3. Ekstraksi Fitur (TF-IDF)

Konfigurasi TF-IDF **dikunci sama untuk seluruh anggota**:

| Parameter | Nilai |
|---|---|
| `analyzer` | `char_wb` (character n-gram dalam batas kata) |
| `ngram_range` | `(2, 4)` |
| `sublinear_tf` | `True` |
| `max_features` | `20.000` |
| `norm` | `l2` (default) |

Character n-gram dipilih karena teks struk rentan salah baca OCR
(`INDOMIE` bisa terbaca `1ND0M1E`); pola huruf tetap mirip meski ada karakter
yang tertukar. TF-IDF di-*fit* hanya pada data latih untuk mencegah kebocoran data.


In [ ]:
vectorizer, x_train = pu.fit_features(train)
x_val = vectorizer.transform(val["text"])
x_test = vectorizer.transform(test["text"])

print(f"Bentuk matriks TF-IDF latih : {x_train.shape}")
print(f"Bentuk matriks TF-IDF uji   : {x_test.shape}")
print(f"Jumlah fitur (n-gram)       : {x_train.shape[1]:,}")
print(f"Kepadatan matriks           : {x_train.nnz / (x_train.shape[0] * x_train.shape[1]):.4%}")


### 4. Pembuatan Model — Multinomial Naive Bayes

Model yang digunakan: **`MultinomialNB`**

```python
model = MultinomialNB(
    alpha=0.03,        # Laplace smoothing; kecil karena TF-IDF sudah sublinear
    fit_prior=True,    # prior kelas mengikuti frekuensi di data latih
)
```

> - **`alpha=0.03`** — Laplace smoothing. Nilai kecil dipilih karena fitur TF-IDF sudah dinormalisasi dan sublinear; smoothing besar justru menghapus sinyal kata kunci produk.
> - **`fit_prior=True`** — probabilitas awal tiap kategori mengikuti distribusi data latih.
>
> Naive Bayes menghitung `log P(kategori) + Σ TF-IDF × log P(fitur | kategori)` dan memilih skor tertinggi. Model ini sangat cepat, cocok untuk inferensi *on-device* di aplikasi Flutter.


In [ ]:
from sklearn.naive_bayes import MultinomialNB

In [ ]:
model = MultinomialNB(
    alpha=0.03,        # Laplace smoothing; kecil karena TF-IDF sudah sublinear
    fit_prior=True,    # prior kelas mengikuti frekuensi di data latih
)

model.fit(x_train, train['kategori'])
print('Model selesai dilatih.')

### 5. Evaluasi

Evaluasi dilakukan pada **data uji** (790 baris) yang produknya tidak pernah
dilihat model. Metrik yang dilaporkan: akurasi, presisi, recall, dan F1-score
(macro), serta confusion matrix 15 kategori.


In [ ]:
hasil = pu.evaluate_model("Multinomial Naive Bayes", model, x_test, test["kategori"])
pu.print_result(hasil)

display(hasil["_confusion"])


Metrik pada data validasi (bukan data uji) sebagai pembanding tambahan:

In [ ]:
hasil_val = pu.evaluate_model("Multinomial Naive Bayes", model, x_val, val["kategori"])
print(f"Validasi -> Akurasi: {hasil_val['Akurasi']:.4f} | "
      f"Macro-F1: {hasil_val['F1-Score (macro)']:.4f}")


#### Penyimpanan Hasil

Metrik disimpan ke `hasil_perbandingan.csv` agar dapat digabungkan dengan hasil
anggota lain pada notebook `perbandingan_model.ipynb`.


In [ ]:
pu.save_result(hasil)
print(f"Metrik disimpan ke: {pu.RESULTS_CSV}")
display(pu.summary_table(hasil))


### 6. Kesimpulan Model

> Naive Bayes unggul pada kecepatan latih/prediksi (sub-detik) dan mudah diekspor ke JSON untuk Flutter. Kelemahannya, asumsi independensi antar n-gram membuatnya kurang tajam pada kategori yang berbagi kata kunci serupa (mis. Makanan vs Minuman).

### 7. Kontribusi Tim

| Anggota | Peran | Kontribusi |
|---|---|---|
| Rakagali | Ketua / Model Naive Bayes | Menyusun pipeline bersama, melatih dan mengevaluasi Multinomial Naive Bayes, menyusun laporan |
| Aldi | Model k-Nearest Neighbors | Melatih dan mengevaluasi kNN, analisis pengaruh nilai k |
| Rijal | Model Support Vector Machine | Melatih dan mengevaluasi SVM, analisis hyperparameter C |
| Noval | Model Random Forest | Melatih dan mengevaluasi Random Forest, analisis jumlah pohon |
| Semua | Kolaborasi | Menyusun dataset, memverifikasi pembagian data, mereview hasil perbandingan model |

